# MarketSensAI on Colab

Explains stock moves from the news, grounded in market data:
live news (Finnhub + yfinance) → every article analyzed by the LLM (vLLM) → stories → price events explained → briefings → RAG → evaluation → charts.

Works in the Colab web UI **and** in VS Code with the Colab extension (*Select Kernel → Colab → New Colab Server*, then pick a GPU).

**Before you start**
1. Pick a GPU runtime. **A100 or L4 recommended**: a full year of news for 3 tickers is ~15,000 articles. A T4 works but is several times slower. Analyses are cached on Drive, so an interrupted run picks up where it stopped.
2. Finnhub key: read from the project's `.env` if present; otherwise from 🔑 **Secrets** (`FINNHUB_API_KEY`, web UI only); otherwise you'll be prompted.
3. The kernel runs on Colab's machine, not your laptop, so the code has to reach it: through GitHub (`REPO_URL`) or a copy of the `marketsensai` folder in Google Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

Clones the latest code from GitHub, so every run uses exactly what's pushed. Set `REPO_URL = ""` to copy the `marketsensai` folder from Google Drive (`DRIVE_REPO`) instead. The last line prints which commit is running.

In [ ]:
import os, shutil
from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/ayushsh314/marketsensai.git"  # "" = copy from DRIVE_REPO instead
DRIVE_REPO = "/content/drive/MyDrive/marketsensai"
OUTPUT_DIR = "/content/drive/MyDrive/MarketSensAI/outputs"  # on Drive so results and the analysis cache survive

%cd /content
shutil.rmtree("/content/marketsensai", ignore_errors=True)  # never run a stale copy from an earlier session
if REPO_URL:
    !git clone -q {REPO_URL} /content/marketsensai
else:
    shutil.copytree(DRIVE_REPO, "/content/marketsensai", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns(".venv", "outputs", "__pycache__"))
%cd /content/marketsensai
!git log -1 --format="Running commit %h: %s (%ci)" 2>/dev/null || echo "Running a copy from Drive (no git history)"

## 2. Install

vLLM brings its own PyTorch build. Everything below runs through the `marketsensai` CLI in a fresh process, so no runtime restart is needed.

pip will print a red "dependency conflicts" block about preinstalled packages this project doesn't use (RAPIDS `cudf`/`cuml`, `google-adk`). That's expected. The last line should say `GPU ok: True`.

In [ ]:
!pip install -q -e ".[vllm,png]"
# vLLM upgrades PyTorch, but Colab's preinstalled torchaudio stays built for the old CUDA version, and
# `transformers` imports it whenever it's installed, which crashes. Nothing here uses it, so remove it.
!pip uninstall -y -q torchaudio
!python -c "import torch, vllm; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU ok:', torch.cuda.is_available(), '| vllm', vllm.__version__)"

In [ ]:
from getpass import getpass

def finnhub_key():
    if os.environ.get("FINNHUB_API_KEY"):
        return os.environ["FINNHUB_API_KEY"]
    for env in (".env", f"{DRIVE_REPO}/.env"):  # synced from your machine through Drive (never committed)
        if os.path.exists(env):
            for line in open(env):
                if line.startswith("FINNHUB_API_KEY="):
                    return line.split("=", 1)[1].strip()
    try:  # Colab web UI secrets; not available in VS Code
        from google.colab import userdata
        return userdata.get("FINNHUB_API_KEY")
    except Exception:
        return getpass("Finnhub API key: ")

os.environ["FINNHUB_API_KEY"] = finnhub_key()

# Qwen2.5-7B in fp16 needs ~16 GB; on smaller GPUs (T4) use the 4-bit AWQ build of the same model.
gpu_mb = int(os.popen("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits").read().split()[0])
MODEL = "Qwen/Qwen2.5-7B-Instruct" if gpu_mb >= 20000 else "Qwen/Qwen2.5-7B-Instruct-AWQ"
print(f"GPU memory: {gpu_mb} MB → {MODEL}")

## 3. Run

- `QUICK_TEST = True` analyzes only 300 articles per ticker, a few minutes end to end. Use it to check everything works before the full run.
- Fetching a year of news takes ~3–4 minutes. `REUSE_CORPUS = True` reruns on the articles saved by the previous run instead.
- Per-article analyses are cached in `OUTPUT_DIR/cache`, so reruns only process articles they haven't seen.
- `--as-of YYYY-MM-DD` pins the end date so a run can be reproduced later; leave it off to use today.

In [ ]:
TICKERS = "AAPL MSFT AMZN"
TIME_RANGES = "1W 1M 6M 1Y"
QUICK_TEST = False    # True = 300 articles per ticker, for a fast check
REUSE_CORPUS = True  # True = reuse the articles saved by the previous run instead of fetching again

corpus = f"{OUTPUT_DIR}/articles.json"
flags = f"--corpus {corpus}" if REUSE_CORPUS and os.path.exists(corpus) else ""
flags += " --max-articles 300" if QUICK_TEST else ""
print("Reusing", corpus if "--corpus" in flags else "nothing: fetching fresh news", "| quick test:", QUICK_TEST)

!marketsensai run --tickers {TICKERS} --time-ranges {TIME_RANGES} --model {MODEL} --output-dir {OUTPUT_DIR} {flags} --png

## 4. Results

The summary includes the controls needed to interpret the headline numbers:
- `control_explained_*`: share of event days the model explains with the real news, with news from a random other week (`shuffled_news`), and with the move's direction mirrored (`flipped_direction`). The real rate should be clearly above both.
- `pooled_spearman_*`: sentiment ↔ abnormal return across all tickers, with a 95% CI and p-value (per-ticker versions are in `per_ticker`).
- `report_numbers_supported` / `reports_passing_number_check`: every number and date in a briefing must appear in its source data.

Rating sheets for the team are in `OUTPUT_DIR/review/` (explanations and briefings to score 1-5, and claim-verification misses to label).

In [ ]:
import json
from IPython.display import Image, display

with open(f"{OUTPUT_DIR}/eval_results.json") as f:
    print(json.dumps(json.load(f)["summary"], indent=2))

# PNGs render the same in the Colab web UI and VS Code; the interactive HTML versions (hover shows
# each price event's explanation) are saved alongside.
viz = f"{OUTPUT_DIR}/visualizations"
for name in sorted(os.listdir(viz)):
    if name.endswith(".png"):
        print(name)
        display(Image(os.path.join(viz, name), width=900))

In [ ]:
with open(f"{OUTPUT_DIR}/pipeline_results.json") as f:
    results = json.load(f)["tickers"]

ticker = TICKERS.split()[0]
print(next(r for r in results[ticker]["reports"] if r["time_range"] == "1M")["report"])

print("\nBiggest moves and their explanations:")
for e in sorted(results[ticker]["event_explanations"], key=lambda e: -e["significance"])[:5]:
    print(f"- {e['date']} {', '.join(e['labels'])}: [{e['primary_driver']}] {e['explanation']}")

## 5. Ask follow-up questions

Searches the articles and the event explanations; reuses the cached embeddings, so it only loads the model.

In [ ]:
!marketsensai ask "What drove AAPL's biggest drop this year?" --ticker AAPL --tickers {TICKERS} --model {MODEL} --output-dir {OUTPUT_DIR}

## 6. Measure speed (optional)

Times the LLM on articles it has never seen: the output folder is new, so the cache is empty. 1,000 articles per ticker is enough for a stable rate. This only analyzes articles, explains events and writes briefings (no RAG, evaluation or charts). Takes several minutes on an A100.

In [ ]:
import json

TIMING_ARTICLES = 1000  # per ticker
TIMING_DIR = f"{OUTPUT_DIR}_timing"
shutil.rmtree(TIMING_DIR, ignore_errors=True)  # an empty cache, so every article is timed

!marketsensai run --tickers {TICKERS} --time-ranges 1Y --model {MODEL} --output-dir {TIMING_DIR} --corpus {corpus} --max-articles {TIMING_ARTICLES} --skip-rag --skip-eval --skip-viz

with open(f"{TIMING_DIR}/pipeline_results.json") as f:
    run = json.load(f)["run"]
rate = run["articles_newly_analyzed"] / run["analysis_seconds"] * 60
manual_min = run["articles"] * 0.375  # the paper's ~30 min per 80 articles
print(f"{run['articles_newly_analyzed']:,} articles analyzed in {run['analysis_seconds']:.0f} s → {rate:,.0f} articles/min")
print(f"Whole pipeline: {run['pipeline_seconds'] / 60:.1f} min vs. ~{manual_min:,.0f} min reading by hand "
      f"({manual_min / (run['pipeline_seconds'] / 60):,.0f}× faster)")

## 7. Package the results

Colab runs on Google's machine, so it can't write to your laptop directly. This bundles everything needed for the analysis into `MyDrive/MarketSensAI/final_results.zip`. In the Colab web UI it also downloads the zip; otherwise download it from drive.google.com. Unzip it into the project's `outputs/final/`.

In [ ]:
FINAL = "/content/final_results"
shutil.rmtree(FINAL, ignore_errors=True)
os.makedirs(FINAL)
for name in ["eval_results.json", "pipeline_results.json", "articles_enriched.csv", "qa_results.json"]:
    shutil.copy(f"{OUTPUT_DIR}/{name}", FINAL)
for folder in ["review", "visualizations"]:
    shutil.copytree(f"{OUTPUT_DIR}/{folder}", f"{FINAL}/{folder}")
timing = f"{OUTPUT_DIR}_timing/pipeline_results.json"  # from section 6, if it was run
if os.path.exists(timing):
    shutil.copy(timing, f"{FINAL}/timing_pipeline_results.json")
!cd /content/marketsensai && git log -1 --format="%H %s" > {FINAL}/commit.txt

zip_path = shutil.make_archive(f"{os.path.dirname(OUTPUT_DIR)}/final_results", "zip", FINAL)
print(f"Saved {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print(f"Automatic download unavailable ({type(e).__name__}); download it from drive.google.com → MyDrive/MarketSensAI/")